# 06 – Model tables

**Goal:** combine all processed data into one table per model.
**Inputs:** `buurt_year_electricity.csv`, `cbs_buurt_panel_2023codes.csv`, `buurt_year_weather.csv`, `buurt_transformer_stress.csv`
**Outputs:** `Data/processed/model_a_table.csv`, `Data/processed/model_b_table.csv`

## Checks
- No duplicate keys in any input. CBS and weather cover all 5,672 consumption buurten; 5,451 also have transformers.
- 72 buurten have transformers but no consumption (likely business/industrial areas); 221 have consumption but no transformer inside (served from a neighbouring buurt).

## Model A – demand growth (panel: buurt × year)
- **Target:** `log(kWh[t+1] / kWh[t])`, only within the same series and for consecutive years → pairs 2019→20 … 2023→24 and 2025→26; 2024→25 excluded (definition break). 2026 = forecast base for 2027.
- **Features (known at t):** level and structure (connections, kWh/connection, 3-phase share, …), lag growth of kWh and connections, CBS drivers of year t, weather of t and t−1 plus HDD change.
- Timing: SJV[t] reflects use in t−1 → growth t→t+1 is explained by the weather change from t−1 to t.
- **Extremes:** 2% of rows have |log growth| > 0.3; 72% of those coincide with >25% change in connections (new building / re-mapping), 57% in buurten < 100 connections. Noise falls with size (std 0.26 for < 50 conn → 0.05 for 1,000+).
- **Decisions:** train/evaluate on buurten with ≥ 50 connections (91.4% of rows, 99.1% of kWh); target clipped at ±0.5 for training (0.52% of rows). Small buurten are still forecast but flagged as low reliability (543 of 5,620 in 2026).

## Model B – stress snapshot (one row per buurt, 5,423 buurten)
- Consumption and connections from the **2025** file (use during 2024, overlaps the peak period; active connections only); consume-only / smart meter / low tariff from 2024; CBS and weather 2024.
- **Relative-to-infrastructure features:** connections per transformer, kWh per kW capacity, kW capacity per connection, solar homes per transformer, feed-in per kW capacity.
- **Targets:** `stress_nbl` (10.2%, classification), `stress_capacity` (6.7%), `util_weighted` (regression, size-independent).
- **Leakage rule:** anything computed from the peaks (`peak_kw_total`, `n_above_*`, `util_*`, `share_summer_peaks`) is never a feature.
- Missing: `p_won_zs` 26.5% (CBS privacy), `g_ele_tr` 5.4% → handled by LightGBM.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path("../Data/processed")

elec    = pd.read_csv(PROCESSED / "buurt_year_electricity.csv")
cbs     = pd.read_csv(PROCESSED / "cbs_buurt_panel_2023codes.csv")
weather = pd.read_csv(PROCESSED / "buurt_year_weather.csv")
stress  = pd.read_csv(PROCESSED / "buurt_transformer_stress.csv")

tables = {"elec": (elec, ["buurtcode", "year"]),
          "cbs": (cbs, ["buurtcode", "year"]),
          "weather": (weather, ["buurtcode", "year"]),
          "stress": (stress, ["buurtcode"])}

for name, (df, keys) in tables.items():
    dups = df.duplicated(keys).sum()
    print(f"{name:8s} shape={df.shape}  buurten={df['buurtcode'].nunique():,}  duplicate keys={dups}")

# do the buurten overlap?
b_elec, b_cbs, b_w, b_s = (set(t[0]["buurtcode"]) for t in tables.values())
print("\nElec buurten also in CBS:    ", len(b_elec & b_cbs))
print("Elec buurten also in weather:", len(b_elec & b_w))
print("Elec buurten also in stress: ", len(b_elec & b_s))
print("Stress buurten NOT in elec:  ", len(b_s - b_elec))

elec     shape=(44762, 12)  buurten=5,672  duplicate keys=0
cbs      shape=(45376, 33)  buurten=5,672  duplicate keys=0
weather  shape=(45376, 10)  buurten=5,672  duplicate keys=0
stress   shape=(5523, 11)  buurten=5,523  duplicate keys=0

Elec buurten also in CBS:     5672
Elec buurten also in weather: 5672
Elec buurten also in stress:  5451
Stress buurten NOT in elec:   72


In [2]:
e = elec.sort_values(["buurtcode", "year"]).copy()
g = e.groupby(["buurtcode", "series"])          # never look across the series break

# next year (target) and previous year (lag feature), within the same series
e["next_year"] = g["year"].shift(-1)
e["next_kwh"]  = g["total_kwh"].shift(-1)
e["prev_year"] = g["year"].shift(1)
e["prev_kwh"]  = g["total_kwh"].shift(1)
e["prev_conn"] = g["connections"].shift(1)

consecutive_next = e["next_year"] == e["year"] + 1
consecutive_prev = e["prev_year"] == e["year"] - 1

e["target_log_growth"] = np.where(consecutive_next, np.log(e["next_kwh"] / e["total_kwh"]), np.nan)
e["lag_log_growth"]    = np.where(consecutive_prev, np.log(e["total_kwh"] / e["prev_kwh"]), np.nan)
e["lag_conn_growth"]   = np.where(consecutive_prev, np.log(e["connections"] / e["prev_conn"]), np.nan)
e = e.drop(columns=["next_year", "next_kwh", "prev_year", "prev_kwh", "prev_conn"])

# weather of year t and of year t-1
wcols = ["temp_mean_c", "hdd_year", "cdd_year", "frost_days", "solar_kwh_m2_year"]
w_t   = weather[["buurtcode", "year"] + wcols].rename(columns={c: c + "_t" for c in wcols})
w_tm1 = weather[["buurtcode", "year"] + wcols].rename(columns={c: c + "_tm1" for c in wcols})
w_tm1["year"] = w_tm1["year"] + 1             # weather of 2020 is "previous year" for row 2021
e = e.merge(w_t, on=["buurtcode", "year"], how="left").merge(w_tm1, on=["buurtcode", "year"], how="left")
e["hdd_change"] = e["hdd_year_t"] - e["hdd_year_tm1"]

# CBS drivers of year t
e = e.merge(cbs, on=["buurtcode", "year"], how="left")

print("Shape:", e.shape)
print("\nRows with a target, per year (t -> t+1):")
print(e.groupby("year")["target_log_growth"].count().to_string())
print("\nTarget (log growth) distribution:")
print(e["target_log_growth"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(3).to_string())

Shape: (44762, 57)

Rows with a target, per year (t -> t+1):
year
2019    5557
2020    5567
2021    5577
2022    5589
2023    5598
2024       0
2025    5609
2026       0

Target (log growth) distribution:
count    33497.000
mean         0.007
std          0.116
min         -2.479
1%          -0.229
5%          -0.085
50%          0.001
95%          0.122
99%          0.367
max          2.744


In [3]:
t = e.dropna(subset=["target_log_growth"]).copy()

# diagnostic only (NOT a feature): connection growth to next year
t = t.merge(
    e[["buurtcode", "year", "connections"]].assign(year=lambda d: d["year"] - 1)
     .rename(columns={"connections": "next_conn"}),
    on=["buurtcode", "year"], how="left")
t["next_conn_growth"] = np.log(t["next_conn"] / t["connections"])

# 1. are extremes mostly small buurten?
t["size_band"] = pd.cut(t["connections"], [0, 50, 100, 250, 500, 1000, 1e9],
                        labels=["<50", "50-100", "100-250", "250-500", "500-1000", "1000+"])
by_size = t.groupby("size_band", observed=True)["target_log_growth"].agg(
    rows="count",
    std="std",
    extreme_pct=lambda s: (s.abs() > 0.3).mean() * 100,
)
print("By buurt size (connections):")
print(by_size.round(3).to_string())

# 2. do extremes come with a big change in connections (new building / re-mapping)?
ext = t[t["target_log_growth"].abs() > 0.3]
print(f"\nExtreme rows (|log growth| > 0.3): {len(ext)} ({100 * len(ext) / len(t):.1f}%)")
print(f"  of which connections also changed > 25%: {100 * (ext['next_conn_growth'].abs() > np.log(1.25)).mean():.1f}%")
print(f"  of which in buurten with < 100 connections: {100 * (ext['connections'] < 100).mean():.1f}%")

By buurt size (connections):
           rows    std  extreme_pct
size_band                          
<50        2952  0.258        7.554
50-100     3127  0.167        5.245
100-250    6432  0.115        2.581
250-500    6415  0.078        1.169
500-1000   7892  0.057        0.418
1000+      6679  0.051        0.225

Extreme rows (|log growth| > 0.3): 676 (2.0%)
  of which connections also changed > 25%: 72.0%
  of which in buurten with < 100 connections: 56.7%


In [4]:
MIN_CONN = 50
CLIP = 0.5

e["train_ok"] = e["connections"] >= MIN_CONN
e["target_clipped"] = e["target_log_growth"].clip(-CLIP, CLIP)

tr = e[e["target_log_growth"].notna()]
tr_ok = tr[tr["train_ok"]]

print(f"Rows with a target:             {len(tr):,}")
print(f"Kept for training (>= {MIN_CONN} conn):  {len(tr_ok):,} ({100 * len(tr_ok) / len(tr):.1f}%)")
print(f"Share of kWh kept:              {100 * tr_ok['total_kwh'].sum() / tr['total_kwh'].sum():.1f}%")
print(f"Targets clipped (of kept rows): {100 * (tr_ok['target_log_growth'].abs() > CLIP).mean():.2f}%")
print(f"Target std before -> after:     {tr_ok['target_log_growth'].std():.3f} -> {tr_ok['target_clipped'].std():.3f}")

print("\nForecast base (2026): buurten =", (e["year"] == 2026).sum(),
      "| low reliability (<50 conn) =", ((e["year"] == 2026) & ~e["train_ok"]).sum())

e.to_csv(PROCESSED / "model_a_table.csv", index=False)
print("\nSaved: model_a_table.csv", e.shape)

Rows with a target:             33,497
Kept for training (>= 50 conn):  30,615 (91.4%)
Share of kWh kept:              99.1%
Targets clipped (of kept rows): 0.52%
Target std before -> after:     0.093 -> 0.078

Forecast base (2026): buurten = 5620 | low reliability (<50 conn) = 543

Saved: model_a_table.csv (44762, 59)


In [5]:
e25 = elec[elec["year"] == 2025][["buurtcode", "connections", "total_kwh", "kwh_per_conn", "share_3phase_pct"]]
e24 = elec[elec["year"] == 2024][["buurtcode", "consume_only_pct", "smart_meter_pct", "low_tariff_pct"]]
c24 = cbs[cbs["year"] == 2024].drop(columns=["year"])
w24 = weather[weather["year"] == 2024][["buurtcode"] + wcols]

b = (stress.merge(e25, on="buurtcode", how="inner")
           .merge(e24, on="buurtcode", how="left")
           .merge(c24, on="buurtcode", how="left")
           .merge(w24, on="buurtcode", how="left"))

# load RELATIVE to the grid infrastructure
b["conn_per_trafo"]        = b["connections"] / b["n_transformers"]
b["kwh_per_kw_capacity"]   = b["total_kwh"] / b["capacity_kw_total"]
b["kw_capacity_per_conn"]  = b["capacity_kw_total"] / b["connections"]
b["solar_homes_per_trafo"] = (b["p_won_zs"] / 100) * b["a_woning"] / b["n_transformers"]
b["feedin_kwh_per_kw_cap"] = b["g_ele_tr"] * b["a_woning"] / b["capacity_kw_total"]

print("Model B shape:", b.shape)
print(f"\nstress_nbl = 1:      {b['stress_nbl'].sum()} ({100 * b['stress_nbl'].mean():.1f}%)")
print(f"stress_capacity = 1: {b['stress_capacity'].sum()} ({100 * b['stress_capacity'].mean():.1f}%)")
print(f"util_weighted: median {b['util_weighted'].median():.2f}, mean {b['util_weighted'].mean():.2f}")

key = ["conn_per_trafo", "kwh_per_kw_capacity", "kw_capacity_per_conn", "solar_homes_per_trafo",
       "feedin_kwh_per_kw_cap", "p_won_zs", "g_ele_tr", "consume_only_pct", "hdd_year"]
print("\n% missing in key features:")
print((b[key].isna().mean() * 100).round(1).to_string())

b.to_csv(PROCESSED / "model_b_table.csv", index=False)
print("\nSaved: model_b_table.csv")


Model B shape: (5423, 59)

stress_nbl = 1:      555 (10.2%)
stress_capacity = 1: 363 (6.7%)
util_weighted: median 0.42, mean 0.42

% missing in key features:
conn_per_trafo            0.0
kwh_per_kw_capacity       0.0
kw_capacity_per_conn      0.0
solar_homes_per_trafo    26.5
feedin_kwh_per_kw_cap     5.4
p_won_zs                 26.5
g_ele_tr                  5.4
consume_only_pct          0.3
hdd_year                  0.0

Saved: model_b_table.csv
